##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 推論

Gemma 是一個輕量級的頂尖開放模型家族，採用與創建 [Gemini](https://deepmind.google/technologies/gemini/#introduction) 模型相同的研發與技術打造而成。Gemma 4 旨在成為全球最具效率的開放權重模型家族。

本文件提供使用 Hugging Face [`transformers`](https://huggingface.co/docs/transformers/installation) 函式庫，以 Gemma 4 進行基本文字推論的指南。內容涵蓋環境設定、模型載入，以及各種文字生成情境，包括單輪提示、結構化多輪對話，以及套用系統指令。

## 安裝 Python 套件

安裝執行 Gemma 模型與發送請求所需的 Hugging Face 函式庫。

In [ ]:
%%bash
# Install PyTorch & other libraries
pip install torch accelerate

# Install the transformers library
pip install "transformers>=5.10.1"


[`Dialog`](https://github.com/google-deepmind/dialog) 是一個用於操作與顯示對話的函式庫。

In [ ]:
%%bash
pip install dialog


## 載入模型

使用 [`transformers`](https://huggingface.co/docs/transformers/installation) 函式庫載入管線（pipeline）

In [ ]:
MODEL_ID = "google/gemma-4-12B-it"

from transformers import pipeline

txt_pipe = pipeline(
    task="text-generation",
    model=MODEL_ID,
    device_map="auto",
    dtype="auto"
)


## 執行文字生成

一旦您已將 Gemma 模型載入並設定為 `pipeline` 物件，就可以向模型發送提示。以下範例程式碼展示了使用 `text_inputs` 參數的基本請求：

In [ ]:
output = txt_pipe(text_inputs="<|turn>user\nRoses are..<turn|>\n<|turn>model\n")
print(output[0]['generated_text'].removesuffix("<turn|>"))


## 使用 Dialog 函式庫

In [ ]:
import dialog
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 512

conv = dialog.Conversation(
    dialog.User("Roses are...")
)
output = txt_pipe(text_inputs=conv.as_text(), return_full_text=False, generation_config=config)
conv += dialog.Model(output[0]['generated_text'].removesuffix("<turn|>"))

print(conv.as_text())
conv.show()


## 使用提示範本

在使用更複雜的提示生成內容時，請使用提示範本來建構您的請求。提示範本讓您能指定來自特定角色的輸入，例如 `user` 或 `model`，是管理與 Gemma 模型多輪聊天互動所需的格式。以下範例程式碼展示如何為 Gemma 建構提示範本：

In [ ]:
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 512

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Write a short poem about the Kraken."},
        ]
    }
]

output = txt_pipe(messages, return_full_text=False, generation_config=config)
print(output[0]['generated_text'].removesuffix("<turn|>"))


## 多輪對話

在多輪設定中，對話記錄會保留為交替出現的 `user` 與 `model` 角色序列。這個累積的清單充當模型的記憶，確保每次新的輸出都以前述對話為依據。

In [ ]:
import dialog
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 512

# User turn #1
conv = dialog.Conversation(
    dialog.User("Write a short poem about the Kraken.")
)

# Model response #1
output = txt_pipe(text_inputs=conv.as_text(), return_full_text=False, generation_config=config)
conv += dialog.Model(output[0]['generated_text'].removesuffix("<turn|>"))

# User turn #2
conv += dialog.User("Now with the Siren.")

# Model response #2
output = txt_pipe(text_inputs=conv.as_text(), return_full_text=False, generation_config=config)
conv += dialog.Model(output[0]['generated_text'].removesuffix("<turn|>"))

print(conv.as_text())
conv.show()


以下是以文字匯出的對話。

In [ ]:
chat_history = conv.as_text(training=True)
print(chat_history)
print("-"*80)

# display as Conversation widget
chat_history


> **Note:** 如果您設定 `training=True`，該對話會被視為完整範例，結尾一定是 `<turn|>`


## 系統指令

使用 `system` 角色來提供系統層級的指令。

In [ ]:
import dialog
from transformers import GenerationConfig
config = GenerationConfig.from_pretrained(MODEL_ID)
config.max_new_tokens = 512

conv = dialog.Conversation(
    dialog.System("Speak like a pirate."),
    dialog.User("Why is the sky blue?")
)

output = txt_pipe(text_inputs=conv.as_text(), return_full_text=False, generation_config=config)
conv += dialog.Model(output[0]['generated_text'].removesuffix("<turn|>"))

print(conv.as_text())
conv.show()


## 摘要
在本指南中，您學會了如何使用 Hugging Face `transformers` 函式庫以 Gemma 4 進行基本文字推論，內容包括：

1. **環境**：設定環境並安裝相依套件。
2. **載入模型**：使用 `pipeline` 抽象層。
3. **推論**：執行基本文字生成。
4. **對話**：使用 [`dialog`](https://github.com/google-deepmind/dialog) 函式庫追蹤對話。
5. **多輪對話**：多輪對話與套用系統指令。